In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(

    passed = RunnablePassthrough(),

    extra = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),

    modified = lambda x: x["num"] + 1
)

In [3]:
runnable.invoke({"num": 1})

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

In [4]:
r = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3)
r.invoke({"num": 1})

{'num': 1, 'mult': 3}

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

C:\Users\user\AppData\Local\Temp\ipykernel_5360\3305729483.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
vectorstore = FAISS.from_texts(
    [
        "테디는 랭체인 주식회사에서 근무를 하였습니다.",
        "셜리는 테디와 같은 회사에서 근무하였습니다.",
        "테디의 직업은 개발자입니다.",
        "셜리의 직업은 디자이너입니다.",
    ],
    embedding = OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini")

def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

retriever_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

retriever_chain.invoke("테디의 직업은 무엇입니까?")

'테디의 직업은 개발자입니다.'

In [7]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [8]:
vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"],
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(
    template
)

model = ChatOpenAI(model="gpt-4o-mini")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [9]:
chain.get_graph().nodes

{'83c9709892b94ba4b478e72caf44078f': Node(id='83c9709892b94ba4b478e72caf44078f', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None),
 '5355a34046624e898983dc3e88319c53': Node(id='5355a34046624e898983dc3e88319c53', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None),
 '900556487c5a4c16898b114c74b19f9e': Node(id='900556487c5a4c16898b114c74b19f9e', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001BF711EAE70>, search_kwargs={}), metadata=None),
 '8d376c7ce8e2460c898b72fa2cf3efbd': Node(id='8d376c7ce8e2460c898b72fa2cf3efbd', name='Passthrough', data=RunnablePassthrough(), metadata=None),
 '7614cd44cad84fd3b8d1582c715b4a83': Node(id='7614cd44cad84fd3b8d1582c715b4a83', name='ChatPromptTemplate', dat

In [10]:
chain.get_graph().edges

[Edge(source='ec3ca8645abb481289e3248bbee3b25e', target='91e41f78771f4787b8881f1a6ce4c43a', data=None, conditional=False),
 Edge(source='91e41f78771f4787b8881f1a6ce4c43a', target='f09c9e391a38410ca03a474c4af5dcf0', data=None, conditional=False),
 Edge(source='ec3ca8645abb481289e3248bbee3b25e', target='5357b32f6b744b58a09d75fc6438d183', data=None, conditional=False),
 Edge(source='5357b32f6b744b58a09d75fc6438d183', target='f09c9e391a38410ca03a474c4af5dcf0', data=None, conditional=False),
 Edge(source='f09c9e391a38410ca03a474c4af5dcf0', target='2cb348ab801d4636bea278beff6422a5', data=None, conditional=False),
 Edge(source='2cb348ab801d4636bea278beff6422a5', target='3f45e52deea04c5ba340f8867cd7e58c', data=None, conditional=False),
 Edge(source='e01edd4bfb6e4a7fb35498ad1fa140e1', target='f69e8bbad4a3437b97f7fa428c42e9ea', data=None, conditional=False),
 Edge(source='3f45e52deea04c5ba340f8867cd7e58c', target='e01edd4bfb6e4a7fb35498ad1fa140e1', data=None, conditional=False)]

In [11]:
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [12]:
chain.get_prompts()

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}'), additional_kwargs={})])]